In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import librosa
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

In [3]:
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

PyTorch: 2.5.1+cu121
CUDA available: True
Using device: cuda


In [4]:
clip_df = pd.read_csv('../clips_metadata.csv',sep=",")

N_MELS = 128
SR = 22050
DURATION = 5
N_FFT = 2048
HOP_LENGTH = 512
NUM_CLASSES = 41

print(f'Total clips: {len(clip_df)}')              # ← clip_df not df
print(f'Classes: {clip_df["species"].nunique()}')   # ← species not emotion

Total clips: 49682
Classes: 41


In [5]:
le = LabelEncoder()
clip_df['label'] = le.fit_transform(clip_df['species'])

In [6]:
class AviaNetDataset(Dataset):
    def __init__(self, clip_df, augment_data=False):
        self.clip_df = clip_df.reset_index(drop=True)
        self.augment_data = augment_data
    def __len__(self):
        return len(self.clip_df)
    def __getitem__(self, idx):
        row = self.clip_df.iloc[idx]
        file_path = row['path']

        y, sr = librosa.load(file_path, sr=SR, duration=DURATION)

        if len(y) < SR * DURATION:
            y = np.pad(y, (0, SR * DURATION - len(y)))

        if self.augment_data:
            y = self.augment(y, sr)

        S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH)
        S_db = librosa.power_to_db(S, ref=np.max)
        S_db = (S_db - S_db.mean()) / (S_db.std() + 1e-8)

        tensor = torch.tensor(S_db, dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(row['label'], dtype=torch.long)

        return tensor, label

    def augment(self, y, sr):
        noise = np.random.randn(len(y)) * 0.005
        y = y + noise
        shift = np.random.randint(0, max(1, sr // 4))
        y = np.roll(y, shift)
        # removed pitch_shift — too slow at this scale
        return y



print('Dataset class defined')


Dataset class defined


In [7]:
train_df, test_df = train_test_split(
    clip_df, test_size=0.2, random_state=42, stratify=clip_df['label']
)

train_df_sample = train_df.sample(n=10000, random_state=42)

train_dataset = AviaNetDataset(train_df_sample, augment_data=True)
test_dataset = AviaNetDataset(test_df, augment_data=False)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

print(f'Train samples: {len(train_dataset)}')
print(f'Test samples:  {len(test_dataset)}')
print(f'Train batches: {len(train_loader)}')
print(f'Test batches:  {len(test_loader)}')

batch_x, batch_y = next(iter(train_loader))
print(f'Batch X shape: {batch_x.shape}')
print(f'Batch y shape: {batch_y.shape}')

Train samples: 10000
Test samples:  9937
Train batches: 157
Test batches:  156
Batch X shape: torch.Size([64, 1, 128, 216])
Batch y shape: torch.Size([64])


In [8]:
class AviaNet(nn.Module):
    def __init__(self,num_classes=NUM_CLASSES):
        super(AviaNet,self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.conv3 =  nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(55296, 512),    # ← fix from 128*16*16
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.classifier(x)
        return x

In [9]:
model = AviaNet(num_classes=NUM_CLASSES).to(device)

with torch.no_grad():
    dummy = torch.zeros(1, 1, 128, 216).to(device)
    out = model.conv1(dummy)
    print(f'After conv1: {out.shape}')
    out = model.conv2(out)
    print(f'After conv2: {out.shape}')
    out = model.conv3(out)
    print(f'After conv3: {out.shape}')
    print(f'Flattened: {out.view(1, -1).shape[1]}')

After conv1: torch.Size([1, 32, 64, 108])
After conv2: torch.Size([1, 64, 32, 54])
After conv3: torch.Size([1, 128, 16, 27])
Flattened: 55296


In [10]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

In [11]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

import torchvision.models as models
efficientnet = models.efficientnet_b0(weights='IMAGENET1K_V1')
print(efficientnet.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)


In [12]:
# Replace final layer for 8 classes
efficientnet.classifier = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(1280, NUM_CLASSES)
)

# Convert grayscale (1 channel) to 3 channels EfficientNet expects
efficientnet.features[0][0] = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False)

# Move to GPU
efficientnet = efficientnet.to(device)

# Freeze all layers except classifier
for param in efficientnet.parameters():
    param.requires_grad = False

# Unfreeze classifier only
for param in efficientnet.classifier.parameters():
    param.requires_grad = True

# Unfreeze first conv too since we changed it
for param in efficientnet.features[0].parameters():
    param.requires_grad = True

print('EfficientNet ready')
print(f'Classifier: {efficientnet.classifier}')

EfficientNet ready
Classifier: Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1280, out_features=41, bias=True)
)


In [ ]:
optimizer_eff = optim.Adam(
    filter(lambda p: p.requires_grad, efficientnet.parameters()),
    lr=0.001
)
scheduler_eff = optim.lr_scheduler.StepLR(optimizer_eff, step_size=10, gamma=0.5)

EPOCHS_EFF = 30
eff_losses = []
eff_accuracies = []

for epoch in range(EPOCHS_EFF):
    efficientnet.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer_eff.zero_grad()
        outputs = efficientnet(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer_eff.step()
        running_loss += loss.item()

    scheduler_eff.step()

    efficientnet.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = efficientnet(X_batch)
            _, predicted = torch.max(outputs, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()

    acc = correct / total
    avg_loss = running_loss / len(train_loader)
    eff_losses.append(avg_loss)
    eff_accuracies.append(acc)

    print(f'Epoch [{epoch+1}/{EPOCHS_EFF}] Loss: {avg_loss:.4f} | Test Acc: {acc:.4f}')

Epoch [1/30] Loss: 3.2219 | Test Acc: 0.0880
Epoch [2/30] Loss: 3.0757 | Test Acc: 0.0750
Epoch [3/30] Loss: 3.0192 | Test Acc: 0.0746
Epoch [4/30] Loss: 2.9792 | Test Acc: 0.0860
Epoch [5/30] Loss: 2.9653 | Test Acc: 0.0677
Epoch [6/30] Loss: 2.9236 | Test Acc: 0.0809
Epoch [7/30] Loss: 2.9094 | Test Acc: 0.0883
Epoch [8/30] Loss: 2.8890 | Test Acc: 0.0835
Epoch [9/30] Loss: 2.8614 | Test Acc: 0.0767
Epoch [10/30] Loss: 2.8717 | Test Acc: 0.0896


In [ ]:
# Unfreeze everything
for param in efficientnet.parameters():
    param.requires_grad = True

# Lower learning rate for fine-tuning
optimizer_eff2 = optim.Adam(efficientnet.parameters(), lr=0.0001)
scheduler_eff2 = optim.lr_scheduler.StepLR(optimizer_eff2, step_size=10, gamma=0.5)

EPOCHS_EFF2 = 70
for epoch in range(EPOCHS_EFF2):
    efficientnet.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer_eff2.zero_grad()
        outputs = efficientnet(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer_eff2.step()
        running_loss += loss.item()

    scheduler_eff2.step()

    efficientnet.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = efficientnet(X_batch)
            _, predicted = torch.max(outputs, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()

    acc = correct / total
    avg_loss = running_loss / len(train_loader)
    print(f'Epoch [{epoch+31}/{100}] Loss: {avg_loss:.4f} | Test Acc: {acc:.4f}')
